# 🧪 03: Ragas Framework Evaluation

**Enterprise Multi-Modal Financial RAG Benchmark**  
**Document:** JPMorgan Chase & Co. 2025 Annual Report  
**Scope:** 20 Easy Benchmark Questions (`GT-E-001` through `GT-E-020`)  
**Framework:** Ragas 0.4.x  
**Models:** Local Ollama `gemma4:26b` (Evaluator LLM) + `qwen3-embedding:4b` (Embeddings)

---

### Standard Ragas Metrics Evaluated:
1. **Faithfulness:** Quantifies whether all claims in the generated response can be inferred directly from the retrieved context (hallucination detection).
2. **Answer Relevancy:** Measures how relevant the generated answer is to the given question.
3. **Context Precision:** Measures whether the ground-truth relevant chunks were ranked higher than irrelevant ones.
4. **Context Recall:** Measures whether the retrieved context contains all the necessary information present in the reference answer.


In [1]:
import sys
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path.cwd()
if BASE_DIR.name == "evaluation":
    BASE_DIR = BASE_DIR.parent
sys.path.insert(0, str(BASE_DIR))

import ragas
from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama, OllamaEmbeddings

from agent.tools import retrieve_chunks, generate_grounded_answer, OLLAMA_URL
from agent.models import SYNTHESIS_MODEL_NAME

print(f"✅ Ragas {ragas.__version__} initialized with local Ollama.")


C:\Users\user\AppData\Local\Temp\ipykernel_4380\1247826605.py:16: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\user\AppData\Local\Temp\ipykernel_4380\1247826605.py:16: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\user\AppData\Local\Temp\ipykernel_4380\1247826605.py:16: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ra

[*] Agent Orchestrator  : Ollama 'qwen3.5:9b' @ http://localhost:11434
✅ Ragas 0.4.3 initialized with local Ollama.


## 1. Configure Local Ollama Models for Ragas


In [2]:
# Evaluator LLM (Gemma 26B)
eval_llm = ChatOllama(
    base_url=OLLAMA_URL,
    model=SYNTHESIS_MODEL_NAME,
    temperature=0.0
)
ragas_llm = LangchainLLMWrapper(eval_llm)

# Evaluator Embeddings (Qwen 4B)
eval_embeddings = OllamaEmbeddings(
    base_url=OLLAMA_URL,
    model="qwen3-embedding:4b"
)
ragas_emb = LangchainEmbeddingsWrapper(eval_embeddings)

print("✅ Ragas local wrappers configured.")


C:\Users\user\AppData\Local\Temp\ipykernel_4380\1204107959.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(eval_llm)


✅ Ragas local wrappers configured.


C:\Users\user\AppData\Local\Temp\ipykernel_4380\1204107959.py:14: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_emb = LangchainEmbeddingsWrapper(eval_embeddings)


## 2. Load Easy Benchmark Questions & Assemble Ragas Dataset


In [3]:
gt_path = BASE_DIR / "evaluation" / "ground_truth_dataset.json"
with open(gt_path, "r", encoding="utf-8") as f:
    gt_data = json.load(f)

easy_questions = [q for q in gt_data["questions"] if q.get("difficulty") == "easy"]

data_samples = {
    "user_input": [],
    "response": [],
    "retrieved_contexts": [],
    "reference": []
}

print(f"Executing RAG pipeline on {len(easy_questions)} Easy questions...")
for idx, q in enumerate(easy_questions, 1):
    query = q["question"]
    ref = q["ground_truth_answer"]
    
    # 1. Retrieve
    raw_context = retrieve_chunks.invoke({"query": query, "limit": 5})
    context_chunks = [c.strip() for c in raw_context.split("---") if c.strip()]
    
    # 2. Synthesize with Groq (Llama 3.3 70B)
    gen_ans = generate_grounded_answer.invoke({
        "question": query,
        "retrieved_evidence": raw_context,
        "provider": "groq",
    })
    
    data_samples["user_input"].append(query)
    data_samples["response"].append(gen_ans)
    data_samples["retrieved_contexts"].append(context_chunks)
    data_samples["reference"].append(ref)
    print(f"[{idx:02d}/20] Captured: {q['id']}")

dataset = EvaluationDataset.from_pandas(pd.DataFrame(data_samples))
print(f"\n✅ Assembled Ragas EvaluationDataset with {len(dataset)} samples.")

Executing RAG pipeline on 20 Easy questions...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[01/20] Captured: GT-E-001
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[02/20] Captured: GT-E-002
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[03/20] Captured: GT-E-003
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[04/20] Captured: GT-E-004
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[05/20] Captured: GT-E-005
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[06/20] Captured: GT-E-006
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[07/20] Captured: GT-E-007
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[08/20] Captured: GT-E-008
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[09/20] Captured: GT-E-009
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[10/20] Captured: GT-E-010
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[11/20] Captured: GT-E-011
[*] Answer

## 3. Run Ragas Evaluation Suite


In [4]:
metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
]

# Set evaluator models
for m in metrics:
    if hasattr(m, "llm"):
        m.llm = ragas_llm
    if hasattr(m, "embeddings"):
        m.embeddings = ragas_emb

results = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_emb
)

results_df = results.to_pandas()
print("✅ Ragas evaluation complete!")
results_df.head()

Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

Exception raised in Job[0]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[9]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[15]: TimeoutError()


KeyboardInterrupt: 

## 4. Aggregate Metric Scorecard


In [ ]:
metric_summary = pd.DataFrame([
    {"Ragas Metric": "Faithfulness (Hallucination Control)", "Score": f"{results_df['faithfulness'].mean():.3f}"},
    {"Ragas Metric": "Answer Relevancy", "Score": f"{results_df['answer_relevancy'].mean():.3f}"},
    {"Ragas Metric": "Context Precision", "Score": f"{results_df['context_precision'].mean():.3f}"},
    {"Ragas Metric": "Context Recall", "Score": f"{results_df['context_recall'].mean():.3f}"},
])
metric_summary


## 5. Visualizing Ragas Metric Distributions


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle("Ragas Metric Distribution Across 20 Easy Questions", fontsize=15, fontweight="bold", y=1.05)

metrics_to_plot = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
colors = ["#10b981", "#3b82f6", "#f59e0b", "#8b5cf6"]

for ax, m, col in zip(axes, metrics_to_plot, colors):
    if m in results_df.columns:
        ax.hist(results_df[m].dropna(), bins=5, color=col, edgecolor="black", alpha=0.8)
        ax.set_title(m.replace("_", " ").title(), fontweight="semibold")
        ax.set_xlabel("Score (0.0 to 1.0)")
        ax.set_ylabel("Count")
        ax.set_xlim(0, 1.05)
        ax.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()
